In [6]:
import yfinance as yf
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import model_from_json

# =====================================================================
# 1. O SEU DICIONÁRIO ORIGINAL (A Chave do Sucesso)
# =====================================================================
tickers = {
    'Bovespa': '^BVSP',        
    'Dolar': 'BRL=X',        
    'SP500': '^GSPC',        
    'Shanghai': '000001.SS', # Bolsa da China
    'Petroleo': 'BZ=F',      
    'Minerio': 'TIO=F',      
    'Ouro': 'GC=F'   
}
time_steps = 7

# 2. Baixar os dados recentes
print("Baixando as variaveis atualizadas...")
df_recente = yf.download(list(tickers.values()), period='15d')['Close']
df_recente.rename(columns={v: k for k, v in tickers.items()}, inplace=True)

# Tratamento para evitar NaNs (Feriados)
df_recente.ffill(inplace=True)
df_recente.dropna(inplace=True)

# Isolar a última janela
ultimos_dias = df_recente.tail(time_steps)
colunas_features = df_recente.columns

print(f"Base de previsão: {ultimos_dias.index[0].strftime('%d/%m/%Y')} até {ultimos_dias.index[-1].strftime('%d/%m/%Y')}")

# =====================================================================
# 3. CARREGAR INTELIGÊNCIA SALVA NO DISCO
# =====================================================================
print("\nCarregando inteligência artificial...")
scaler_X_carregado = joblib.load('modelos/scaler_X.pkl')
scaler_y_carregado = joblib.load('modelos/scaler_y.pkl')

with open('modelos/modelo_gru.json', 'r') as json_file:
    modelo_gru_carregado = model_from_json(json_file.read())
modelo_gru_carregado.load_weights('modelos/modelo_gru.weights.h5')

# =====================================================================
# 4. A PREVISÃO (Agora o Scaler vai reconhecer tudo!)
# =====================================================================
X_futuro = ultimos_dias[colunas_features]

# Como os nomes das colunas agora batem 100% com o treino, o transform vai funcionar de primeira
X_futuro_scaled = scaler_X_carregado.transform(X_futuro)
X_futuro_reshaped = X_futuro_scaled.reshape(1, time_steps, X_futuro.shape[1])

previsao_scaled = modelo_gru_carregado.predict(X_futuro_reshaped, verbose=0)
previsao_real = scaler_y_carregado.inverse_transform(previsao_scaled)

print(f"\n🚀 PREVISÃO DA BOVESPA PARA O PRÓXIMO DIA ÚTIL: {previsao_real[0][0]:.2f} pontos 🚀")

Baixando as variaveis atualizadas...


[*********************100%***********************]  7 of 7 completed


Base de previsão: 11/05/2026 até 19/05/2026

Carregando inteligência artificial...

🚀 PREVISÃO DA BOVESPA PARA O PRÓXIMO DIA ÚTIL: 172328.84 pontos 🚀
